In [1]:
import src.database.scripts.sql as sql 

import requests
from datetime import datetime, timedelta, timezone
import numpy as np

In [234]:
conn = sql.connect_pc()
cursor = conn.cursor()

query = """
    SELECT * FROM item_data
    WHERE type LIKE '%Misc'
    AND name NOT LIKE '%Gold Coin%'
    AND name NOT LIKE '%Volcano%'
    AND (name NOT LIKE 'Ruby%' and type LIKE '%Misc')
    AND name NOT LIKE '%Diamond%'
    AND name NOT LIKE '%Saphhire%'
    AND name NOT LIKE '%Emerald%';
    """

cursor.execute(query)
sql_fetch = cursor.fetchall()

In [249]:
from_date = (datetime.now(timezone.utc) - timedelta(minutes=3)).strftime("%Y-%m-%dT%H:%M:%SZ")
def get_url(item_id, from_data, page):
    return f"""https://api.darkerdb.com/v1/market?item_id={item_id}&from={from_date}&limit=50&page={page}"""


with requests.session() as ses:
    output = [('name', 'rarity', 'amount', 'price', 'price_per_unit', 'created', 'sold')]
    for item_id in sql_fetch:
        page = 1
        fetch = ses.get(get_url(item_id[0], from_date, page)).json()
        while fetch['pagination']['count'] != 0:
            for item in fetch['body']:
                output.append((item['item'], item['rarity'], item['quantity'], item['price'], item['price_per_unit'], item['created_at'], item['sold_at']))
            print(f"{item_id[0]:^20} page:{page:<10}", end='\r')
            page += 1
            url = get_url(item_id[0], from_date, page)
            fetch = ses.get(url).json()
            output[1:] = sorted(output[1:], key=lambda x: (x[3]), reverse=True)
        # for row in output:
        #     print(f"{row[0]:<30} {row[1]:>10} {row[2]:>10} {row[3]:>10} {row[4]:>15} {row[5]:>25} {str(row[6]):>25}")
    

In [247]:
output[1:] = sorted(output[1:], key=lambda x: (x[3]), reverse=True)
for row in output:
    print(f"{row[0]:<30} {row[1]:>10} {row[2]:>10} {row[3]:>10} {row[4]:>15} {row[5]:>25} {str(row[6]):>25}")

name                               rarity     amount      price  price_per_unit                   created                      sold
Warlord's Armor Shard                Rare          1      30000           30000      2026-04-27T11:00:56Z                      None
Ectoplasm                        Uncommon          3      14331            4777      2026-04-27T10:58:35Z                      None
Mysterious Potion               Legendary          1      12222           12222      2026-04-27T11:04:11Z                      None
Ectoplasm                        Uncommon          1       6000            6000      2026-04-27T10:58:39Z                      None
Arcane Essence                     Unique          1       5300            5300      2026-04-27T10:58:33Z                      None
Arcane Essence                     Unique          1       5250            5250      2026-04-27T11:00:13Z                      None
Arcane Essence                     Unique          1       5111            5